In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import os
import time
import numpy as np
from itertools import chain
import pickle
from glob import glob
from pathlib import Path
from copy import deepcopy

from multiprocessing import Pool

from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from utils.write_output import write_array2txt
from utils.math_methods import project_point_onto_plane

from cylinder_extraction.utils import detect_objects

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## For multiple directories

In [3]:
field = 'Tank43 Loose Registration'
pack_num = 8 # number of parallel cores
data_dir = Path(f'D:\REGISTRATION DATA\External/{field}')

In [4]:
with open('D:\\REGISTRATION DATA\\External\\Tank43 Loose Registration\\Tank43_Gibsontk43_Scan_003\\Tank43_Gibsontk43_Scan_003.pkl', 'rb') as f:
    s3 = pickle.load(f)

In [6]:
s3.refined_cylinders

[{'station': {'name': 'Tank43_Gibsontk43_Scan_003'}, 'radius': 21.58805, 'axis': "{'position': array([ -1.8231803 , -77.73188   ,  -0.32992423], dtype=float32), 'vector': array([-0.00222373, -0.00420468,  0.9999887 ], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\Tank43 Loose Registration\\Tank43_Gibsontk43_Scan_003\\Tank43_Gibsontk43_Scan_003 cluster_12.pcd'},
 {'station': {'name': 'Tank43_Gibsontk43_Scan_003'}, 'radius': 12.343697, 'axis': "{'position': array([ 0.46481815, 35.38127   , -0.09637469], dtype=float32), 'vector': array([9.906021e-04, 2.694704e-03, 9.999959e-01], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\Tank43 Loose Registration\\Tank43_Gibsontk43_Scan_003\\Tank43_Gibsontk43_Scan_003 cluster_13.pcd'},
 {'station': {'name': 'Tank43_Gibsontk43_Scan_003'}, 'radius': 11.8167925, 'axis': "{'position': array([ -4.932213  , -31.929632  ,   0.54400593], dtype=float32), 'vector': array([0.01365592, 0.01487021, 0.99979615], dtype=floa

In [29]:
s3.read_points()

D:\REGISTRATION DATA\External\Tank43 Loose Registration Tank43_Gibsontk43_Scan_003.e57
[0.899306   0.00547561 0.00267949 0.43727743]
[-12.45816555  -7.22524302 667.71954765]
fields
['sphericalRange', 'sphericalAzimuth', 'sphericalElevation', 'intensity', 'rowIndex', 'columnIndex', 'sphericalInvalidState']
theta -3.141592502593994 3.141592502593994
phi -1.0227746963500977 1.5704925060272217


In [31]:
def foo():
    return np.random.rand(2,3)

In [33]:
foo().reshape((1,6)).shape

(1, 6)

In [30]:
s3.estimate_ground_z()

665.9875644838538

In [11]:
cyl = s3.refined_cylinders[-1]

In [12]:
cyl.read_points()

In [22]:
cyl.points.max(axis=0)

array([  5.81792879,  -9.1695528 , 676.87823486])

In [23]:
cyl.points.min(axis=0)

array([ -8.1808157 , -25.16935921, 666.05755615])

In [24]:
cyl

{'station': {'name': 'Tank43_Gibsontk43_Scan_003'}, 'radius': 11.874972, 'axis': "{'position': array([ -4.205124, -31.746956,   0.517199], dtype=float32), 'vector': array([0.01266783, 0.01454201, 0.99981403], dtype=float32)}", 'ptc_filepath': 'D:\\REGISTRATION DATA\\External\\Tank43 Loose Registration\\Tank43_Gibsontk43_Scan_003\\Tank43_Gibsontk43_Scan_003 cluster_14.pcd'}

In [27]:
cyl.write_points(filename='s003_c14.pcd', center=True, data_type='pcd')

write to D:\REGISTRATION DATA\External\Tank43 Loose Registration\Tank43_Gibsontk43_Scan_003\s003_c14.pcd: (2293612, 3) points


In [13]:
fit_info = cyl.fit(refined=False)

10000
2189226 inlier points


In [16]:
fit_info['cyl_params']

array([-5.7655864e+00, -3.0474131e+01,  4.5818821e-01,  1.1946074e+01,
        1.5087560e-02,  1.2590182e-02,  9.9980688e-01], dtype=float32)

In [17]:
d = cyl.axis.distance_to_point(cyl.points)

In [24]:
import plotly.express as px

In [21]:
np.mean(d)

11.890472825522137

In [49]:
field = 'JB-448 26ft EXT Faro'
data_dir = Path(f'D:\REGISTRATION DATA\External/{field}')

In [50]:
with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'rb') as f:
    bundle = pickle.load(f)

In [51]:
for s in bundle.stations:
    print(s.name, s.z_ground)

Jb-448 Scan_FARO_LS03550_088 -20.429303513909886
Jb-448 Scan_FARO_LS03550_089 -21.900154306191766
Jb-448 Scan_FARO_LS03550_090 -21.904379384551614
Jb-448 Scan_FARO_LS03550_091 -21.609031463936454
Jb-448 Scan_FARO_LS03550_092 -21.33092342066972
Jb-448 Scan_FARO_LS03550_093 -20.609584237562405
Jb-448 Scan_FARO_LS03550_094 -21.909714723096428
Jb-448 Scan_FARO_LS03550_095 -20.4091129354561
Jb-448 Scan_FARO_LS03550_096 -18.567001577278557


In [47]:
s = bundle.stations[-5]
s

{'name': 'Tank43_Gibsontk43_Scan_015'}

In [48]:
s.read_points()
s.estimate_ground_z()
s.z_ground

D:\REGISTRATION DATA\External\Tank43 Loose Registration Tank43_Gibsontk43_Scan_015.e57
[ 8.33784587e-01 -1.16324092e-03  6.89908055e-04 -5.52088247e-01]
[ 21.28324491 -30.92220826 668.14636593]
fields
['sphericalRange', 'sphericalAzimuth', 'sphericalElevation', 'intensity', 'rowIndex', 'columnIndex', 'sphericalInvalidState']
theta -3.141592502593994 3.141592025756836
phi -1.0215489864349365 1.5704926252365112


666.4599897536365

In [43]:
s18.estimate_ground_z()

666.3284845248124

In [44]:
s18.z_ground

666.3284845248124

In [41]:
# two step
# coarse estimate: every 0.5 meter
step_rough = 0.5
z_max = s18.ptc[:,2].max()
z_min = s18.ptc[:,2].min()
hist, bin_edges = np.histogram(s18.ptc, bins=np.arange(z_min, z_max, step_rough))
# the ground is determined with respect to the bin with the most number of points, i.e. z
max_ind = np.argmax(hist)
# fine estimate
step_fine = 0.02
z_min = bin_edges[max_ind]
z_max = bin_edges[max_ind + 1]
hist, bin_edges = np.histogram(s18.ptc, bins=np.arange(z_min, z_max, step_fine))
# the ground is determined with respect to the bin with the most number of points, i.e. z
z_ground = bin_edges[np.argmax(hist)]

In [33]:
np.argmax(hist)

108

In [42]:
z_ground

666.3284845248124

In [30]:
z_ground

668.0284845248109

In [37]:
hist, bin_edges = np.histogram(s18.ptc, bins=np.arange(z_min, z_max, 0.5))

In [40]:
n = 10
px.line(x=bin_edges[:n], y=hist[:n])

In [9]:
bundle.find_uncertain_stations()

KeyboardInterrupt: 